In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, lit, row_number, sum as _sum, count, avg,
    round as _round, dense_rank, date_format, year, month, dayofweek,
    quarter, weekofyear, when, datediff, current_date, max as _max,
    min as _min, countDistinct, expr, monotonically_increasing_id
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "dev", "Catalog Name")
CATALOG = dbutils.widgets.get("catalog")
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

In [0]:
def write_gold(df, table_name, mode="overwrite"):
    """Write a dataframe as a Gold Delta table."""
    target = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    (df.write
     .format("delta")
     .mode(mode)
     .option("overwriteSchema", "true")
     .saveAsTable(target))
    cnt = spark.table(target).count()
    print(f"  {table_name}: {cnt:,} rows written")
    return cnt

In [0]:
df_customers = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers")
df_products  = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_products")
df_orders    = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders")
df_items     = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_order_items")
df_enriched  = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched")

In [0]:
dim_customers = (df_customers
    .select(
        col("customer_id"),
        col("name").alias("customer_name"),
        col("city"),
        col("state"),
        col("signup_date"),
    )
    # Enrich with order metrics per customer
    .join(
        df_orders.groupBy("customer_id").agg(
            _min("order_date").alias("first_order_date"),
            _max("order_date").alias("last_order_date"),
            count("order_id").alias("total_orders"),
            _sum("total_amount").alias("lifetime_spend"),
            _round(avg("total_amount"), 2).alias("avg_order_value"),
        ),
        "customer_id",
        "left"
    )
    .withColumn("days_since_last_order",
        datediff(current_date(), col("last_order_date"))
    )
    # Simple RFM segment based on order count and recency
    .withColumn("customer_segment",
        when(col("total_orders").isNull(), "No Orders")
        .when((col("total_orders") >= 10) & (col("days_since_last_order") <= 30), "Champion")
        .when((col("total_orders") >= 5) & (col("days_since_last_order") <= 60), "Loyal")
        .when((col("total_orders") >= 3) & (col("days_since_last_order") <= 90), "Regular")
        .when(col("days_since_last_order") > 180, "At Risk")
        .otherwise("Occasional")
    )
    .withColumn("_gold_processed_at", current_timestamp())
)

write_gold(dim_customers, "dim_customers")

In [0]:
dim_products = (df_products
    .select(
        col("product_id"),
        col("product_name"),
        col("category"),
        col("price").alias("list_price"),
    )
    # Enrich with sales metrics per product
    .join(
        df_items.groupBy("product_id").agg(
            _sum("quantity").alias("total_qty_sold"),
            _sum(col("quantity") * col("price")).alias("total_revenue"),
            countDistinct("order_id").alias("total_orders"),
            _round(avg("price"), 2).alias("avg_selling_price"),
        ),
        "product_id",
        "left"
    )
    .withColumn("product_rank",
        dense_rank().over(Window.orderBy(col("total_revenue").desc()))
    )
    .withColumn("_gold_processed_at", current_timestamp())
)

write_gold(dim_products, "dim_products")


In [0]:
from pyspark.sql.functions import explode, sequence, to_date as _to_date

# Get date range from orders
date_range = df_orders.select(
    _min("order_date").alias("min_date"),
    _max("order_date").alias("max_date")
).collect()[0]

dim_date = (spark
    .sql(f"SELECT explode(sequence(DATE'{date_range.min_date}', DATE'{date_range.max_date}', INTERVAL 1 DAY)) as date")
    .select(
        col("date"),
        expr("CAST(date_format(date, 'yyyyMMdd') AS INT)").alias("date_key"),
        year("date").alias("year"),
        quarter("date").alias("quarter"),
        month("date").alias("month"),
        date_format("date", "MMMM").alias("month_name"),
        weekofyear("date").alias("week_of_year"),
        dayofweek("date").alias("day_of_week"),
        date_format("date", "EEEE").alias("day_name"),
        when(dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
        date_format("date", "yyyy-MM").alias("year_month"),
        date_format("date", "yyyy-'Q'q").alias("year_quarter"),
    )
)

write_gold(dim_date, "dim_date")


In [0]:
fact_sales = (df_enriched
    .select(
        col("order_item_id"),
        col("order_id"),
        expr("CAST(date_format(order_date, 'yyyyMMdd') AS INT)").alias("date_key"),
        col("customer_id"),
        col("product_id"),
        col("order_date"),
        col("order_status"),
        col("quantity"),
        col("unit_price"),
        (col("quantity") * col("unit_price")).alias("line_total"),
        col("order_total"),
        col("customer_name"),
        col("customer_city"),
        col("customer_state"),
        col("product_name"),
        col("product_category"),
    )
    .withColumn("_gold_processed_at", current_timestamp())
)
write_gold(fact_sales, "fact_sales")

In [0]:
agg_revenue_by_state = (fact_sales
    .groupBy("customer_state")
    .agg(
        _round(_sum("line_total"), 2).alias("total_revenue"),
        count("order_item_id").alias("total_items_sold"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        _round(avg("line_total"), 2).alias("avg_item_value"),
    )
    .withColumn("state_rank",
        dense_rank().over(Window.orderBy(col("total_revenue").desc()))
    )
    .orderBy("state_rank")
    .withColumn("_gold_processed_at", current_timestamp())
)

write_gold(agg_revenue_by_state, "agg_revenue_by_state")


In [0]:
agg_top_products = (fact_sales
    .groupBy("product_id", "product_name", "product_category")
    .agg(
        _round(_sum("line_total"), 2).alias("total_revenue"),
        _sum("quantity").alias("total_qty_sold"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_buyers"),
        _round(avg("unit_price"), 2).alias("avg_unit_price"),
    )
    .withColumn("product_rank",
        dense_rank().over(Window.orderBy(col("total_revenue").desc()))
    )
    .withColumn("category_rank",
        dense_rank().over(
            Window.partitionBy("product_category").orderBy(col("total_revenue").desc())
        )
    )
    .orderBy("product_rank")
    .withColumn("_gold_processed_at", current_timestamp())
)

write_gold(agg_top_products, "agg_top_products")


In [0]:
agg_sales_daily = (fact_sales
    .groupBy("order_date")
    .agg(
        _round(_sum("line_total"), 2).alias("daily_revenue"),
        count("order_item_id").alias("items_sold"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        _round(avg("line_total"), 2).alias("avg_item_value"),
    )
    .orderBy("order_date")
    .withColumn("_gold_processed_at", current_timestamp())
)

write_gold(agg_sales_daily, "agg_sales_daily")


In [0]:
agg_sales_monthly = (fact_sales
    .withColumn("year_month", date_format("order_date", "yyyy-MM"))
    .withColumn("year", year("order_date"))
    .withColumn("month", month("order_date"))
    .groupBy("year", "month", "year_month")
    .agg(
        _round(_sum("line_total"), 2).alias("monthly_revenue"),
        count("order_item_id").alias("items_sold"),
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("unique_customers"),
        _round(avg("line_total"), 2).alias("avg_item_value"),
    )
    .orderBy("year", "month")
    .withColumn("_gold_processed_at", current_timestamp())
)

write_gold(agg_sales_monthly, "agg_sales_monthly")



In [0]:
gold_tables = [
    "dim_customers", "dim_products", "dim_date",
    "fact_sales",
    "agg_revenue_by_state", "agg_top_products",
    "agg_sales_daily", "agg_sales_monthly"
]

print(f"\n{'='*70}")
print(f"  GOLD LAYER SUMMARY")
print(f"{'='*70}")
print(f"  {'Table':<30s} | {'Rows':>10s} | {'Columns':>8s}")
print(f"  {'-'*30}-+-{'-'*10}-+-{'-'*8}")

for t in gold_tables:
    full = f"{CATALOG}.{GOLD_SCHEMA}.{t}"
    df = spark.table(full)
    print(f"  {t:<30s} | {df.count():>10,} | {len(df.columns):>8}")

print(f"{'='*70}")
print(f"  Total Gold tables: {len(gold_tables)}")
